# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# DONE: Import the necessary libs
# For example: 
import os
import json
import chromadb
from tavily import TavilyClient
from pydantic import BaseModel

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [4]:
# DONE: Load environment variables
load_dotenv(find_dotenv(), override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
# DONE: Create retrieve_game tool
# It should use chroma client and collection you created
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

@tool
def retrieve_game(query: str):
    """Semantic search: Finds most results in the vector DB

    args:
    - query: a question about game industry.
    """
    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=['documents', 'metadatas', 'distances']
    )
    output = []
    for doc, metadata, distance in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        output.append({
            'Platform': metadata.get('Platform'),
            'Name': metadata.get('Name'),
            'YearOfRelease': metadata.get('YearOfRelease'),
            'Description': metadata.get('Description'),
            'Distance': distance,
            'Document': doc,
        })
    return output

#### Evaluate Retrieval Tool

In [6]:
# DONE: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class EvaluationReport(BaseModel):
    useful: bool
    description: str

@tool
def evaluate_retrieval(question: str, retrieved_docs: list[dict]):
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    """
    llm = LLM(model="gpt-4o-mini")
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n"
        f"Retrieved docs: {retrieved_docs}"
    )
    response = llm.invoke(input=prompt, response_format=EvaluationReport)
    parser = PydanticOutputParser(model_class=EvaluationReport)
    return parser.parse(response).model_dump()

#### Game Web Search Tool

In [7]:
# DONE: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str):
    """Search the web for game industry questions.

    args:
    - question: a question about game industry.
    """
    return tavily_client.search(query=question, max_results=3)

### Agent

In [8]:
# DONE: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

tools = [retrieve_game, evaluate_retrieval, game_web_search]
instructions = "You are UdaPlay, an AI research agent for the video game industry. Use the available tools when needed."
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=tools,
    temperature=0.0,
)

In [9]:
# DONE: Invoke your agent
run = agent.invoke("When Pokémon Gold and Silver was released?")
print(run.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Pokémon Gold and Silver were released in Japan on November 21, 1999. They were later released in North America on October 15, 2000, and in Europe on April 6, 2001.


### (Optional) Advanced

In [10]:
# DONE: Update your agent with long-term memory
from lib.memory import LongTermMemory
from lib.vector_db import VectorStoreManager

long_term_memory = LongTermMemory(VectorStoreManager(OPENAI_API_KEY))
agent.long_term_memory = long_term_memory
# DONE: Convert the agent to be a state machine, with the tools being pre-defined nodes
state_machine = agent.workflow
print(type(state_machine).__name__)

StateMachine
